# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Target = "Transported"

y = spaceship["Transported"].astype(int)
X = spaceship.drop(columns=["Transported", "PassengerId", "Name"])  # drop non-informative


# Separate categorical and numeric features

num_features = X.select_dtypes(include=['int64','float64']).columns
cat_features = X.select_dtypes(include=['object','bool']).columns


# Pipelines for scaling + imputing

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)


# Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [4]:
model = RandomForestClassifier(random_state=42, n_jobs=-1)

pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

- Evaluate your model

In [5]:
pipe.fit(X_train, y_train)
y_pred_base = pipe.predict(X_test)

print("=== Baseline (default hyperparameters) ===")
print("Accuracy :", accuracy_score(y_test, y_pred_base))
print("F1-score :", f1_score(y_test, y_pred_base))

# For AUC we need probabilities; handle gracefully if not available
try:
    y_proba_base = pipe.predict_proba(X_test)[:, 1]
    print("ROC AUC  :", roc_auc_score(y_test, y_proba_base))
except Exception:
    pass
print()

=== Baseline (default hyperparameters) ===
Accuracy : 0.7866589994249569
F1-score : 0.7826596367896895
ROC AUC  : 0.8629561580342546



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [6]:
param_grid = {
    "model__n_estimators": [200, 400, 800],
    "model__max_depth": [None, 10, 20, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

- Run Grid Search

In [7]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",   # you can try "f1" too
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)

print("=== Grid Search Results ===")
print("Best CV Score:", grid.best_score_)
print("Best Params :", grid.best_params_)
print()

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
=== Grid Search Results ===
Best CV Score: 0.7959469146465719
Best Params : {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 10, 'model__n_estimators': 200}



- Evaluate your model

In [8]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("=== Test Set Performance (Tuned) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
try:
    y_proba = best_model.predict_proba(X_test)[:, 1]
    print("ROC AUC  :", roc_auc_score(y_test, y_proba))
except Exception:
    pass
print()
print("Classification Report:\n", classification_report(y_test, y_pred))

=== Test Set Performance (Tuned) ===
Accuracy : 0.7981598619896493
F1-score : 0.8011331444759207
ROC AUC  : 0.8750251326740637

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.79      0.80       863
           1       0.80      0.81      0.80       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739

